<a href="https://colab.research.google.com/github/Dolores-Xu/Dolores-Xu/blob/main/%E2%80%9CM1_Assignment_RAG_Chatbot_Apple_10K_Report_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 Coding Assignment: AAPL 10-K Report RAG Chatbot

In this assignment, you will build a simple AI chatbot using Apple's 10-K report and Retrieval Augmented Generation (RAG). We will leverage the power of OpenAI API, LangChain, and FAISS (Facebook AI Similarity Search) to create a RAG chatbot similar to what we've explored in class. This chatbot will be capable of answering questions based on the information contained within the 10-K report.

## Objectives
By completing this assignment, you will achieve the following:
- Gain practical experience in implementing a Retrieval Augmented Generation (RAG) system.
- Learn how to utilize the OpenAI API for embedding generation and chatbot interactions.
- Understand the process of retrieving relevant information from a document using FAISS.
- Develop a functional chatbot capable of providing answers based on a specific knowledge source.

## Instructions
1. Follow the step-by-step guide below to install necessary packages, download data, and build your chatbot.
2. Ensure you have an OpenAI API key and replace placeholders accordingly.
3. Experiment with different queries to test the capabilities of your chatbot.
4. In the summary section at the end, document your observations and answer the provided questions.

Let's begin building your RAG chatbot!



## Step 1: Install Required Packages
- Execute the following code cell to install the necessary packages:

In [ ]:
# freezing the coding environment to these specific versions
!pip install sec-edgar-downloader==5.0.3 langchain==0.3.23 langchain_openai==0.3.14 langchain_community==0.3.21 openai==1.72.0 faiss-cpu==1.10.0 --quiet

## Step 2: Mount Google Drive and Load OpenAI Credential
* Mount your Google Drive to access necessary files.
* Read the OpenAI key and set environment variable


In [ ]:
#Mount the google drive
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

# Updated path checking logic
pth = '/content/drive/MyDrive/'
key_file_path = os.path.join(pth, "api/openai")

if os.path.exists(key_file_path):
    with open(key_file_path, "r") as f:
        key = f.read().strip()
    os.environ["OPENAI_API_KEY"] = key
    print("Successfully loaded OpenAI API Key and set environment variable.")
else:
    print(f"Error: The file {key_file_path} was not found. Please check your Google Drive path.")

Mounted at /content/drive
Successfully loaded OpenAI API Key and set environment variable.


##Step 3. Download AAPL 10-K report from SEC Edgar database

[sec_edgar_downloader](https://sec-edgar-downloader.readthedocs.io/en/latest/)

In [ ]:
# 1. Install precise versions to bypass the pyrate-limiter 3.7+ incompatibility
!pip install "sec-edgar-downloader==5.0.3" "pyrate-limiter==3.6.1" --quiet

# 2. Force restart logic to ensure the correct versions are loaded into memory
import os
if not os.environ.get('RESTARTED_V5_FIXED'):
    os.environ['RESTARTED_V5_FIXED'] = '1'
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

from sec_edgar_downloader import Downloader
import os

# REQUIRED: SEC compliance identity
company_name = "Individual"
email_address = "user@example.com"

# Create storage directory
os.makedirs("data/sec-edgar-filings", exist_ok=True)

# Initialize with User-Agent (v5 API style)
dl = Downloader(company_name, email_address, "data/sec-edgar-filings")

# Download the 10-K for AAPL
company_ticker = "AAPL"
dl.get("10-K", company_ticker, limit=1, download_details=True)

print("10-K report downloaded successfully into 'data/sec-edgar-filings'.")

10-K report downloaded successfully into 'data/sec-edgar-filings'.


##Step 4. Parsing with BeautifulSoup

The report we got is in html format and we need to parse it. BeautifulSoup comes in handy.

[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)

In [ ]:
import os
from bs4 import BeautifulSoup

def preprocess_10k_files(download_dir):
    text_data = []
    for root, dirs, files in os.walk(download_dir):
        for file in files:
            if file.lower().endswith((".html", ".htm")):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        soup = BeautifulSoup(f, 'html.parser')
                        content = soup.get_text(separator=' ', strip=True)
                        if content:
                            text_data.append(content)
                            print(f"Successfully parsed: {file}")
                except Exception as e:
                    print(f"Error reading {file}: {e}")
    return text_data

download_dir = "data/sec-edgar-filings"
text_data = preprocess_10k_files(download_dir)
print(f"Total documents found: {len(text_data)}")

Successfully parsed: primary-document.html
Total documents found: 1


In [ ]:
# Debugging: List all files in the directory to see what was actually downloaded
import os
for root, dirs, files in os.walk("data/sec-edgar-filings"):
    for file in files:
        print(os.path.join(root, file))

# Also show the current text_data status
print(f"Current text_data length: {len(text_data)}")

data/sec-edgar-filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/primary-document.html
data/sec-edgar-filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt
Current text_data length: 1


## Step 5: Split the Text for Embedding
- Split the extracted text into smaller chunks for efficient embedding generation.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=100
    )
    return splitter.split_text(text)

In [ ]:
if len(text_data) > 0:
    text_chunks = split_text(text_data[0])
    print(f"Successfully split into {len(text_chunks)} chunks.")
else:
    print("Error: text_data is empty. Please check if the 10-K HTML file exists in the directory.")

Successfully split into 117 chunks.


## Step 6: Generate Embeddings
Use a pre-trained model (like OpenAI's text-embedding-ada-002 model or other embedding models) to convert your text data into vector embeddings. Each text entry will be represented as a vector of numbers.

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings

# 使用 .get() 避免 KeyError，并检查 key 是否存在
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    print("错误：未检测到 OPENAI_API_KEY。请确保已经成功运行了 Step 2 单元格。")
else:
    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=api_key,
        # 填入你截图里的第二个 URL
        base_url="https://yinli.one/v1"
    )
    print("Embedding 模型 (embedding_model) 已成功初始化。")

Embedding 模型 (embedding_model) 已成功初始化。


## Step 7: Setup Vectorstore using FAISS(Facebook AI Similarity Search)

In the cell below, we demonstrate how to use the faiss package to create a FAISS Index directly from colab

FAISS (Facebook AI Similarity Search) is a python library for searching and clustering vectors. Recall that we use embedded word vectors to store knowledge in text AI applications. FAISS, with its similarity functionality, helps us retreive knowledge effectively

For more information on Facebook AI Similarity Search, see: https://github.com/facebookresearch/faiss/wiki and for a tutorial on FAISS, see https://github.com/facebookresearch/faiss/wiki/Getting-started

In [ ]:
from langchain_community.vectorstores import FAISS
import faiss

# Step 3: Embed the Text and Store in FAISS
def embed_and_store(text_chunks, embedding_model, index_name):
    faiss_index = FAISS.from_texts(text_chunks, embedding_model)
    faiss_index.save_local(index_name)
    print(f"FAISS index saved at {index_name}")
    return faiss_index

In [ ]:
# 使用 Step 6 中定义的变量名 embedding_model
faiss_index = embed_and_store(text_chunks, embedding_model, "data/aapl_10k_faiss_index")

FAISS index saved at data/aapl_10k_faiss_index


## Step 8: Connect Chatbot to Knowledge Base
* Implement the retrieval function to fetch relevant information from the FAISS index based on user queries.
* We've built a fully-fledged knowledge base. Now it's time to connect that knowledge base to our chatbot.
*To do that, I've prepared the retrieval function that based on the query, retrives the top 3 similar text chunks from the 10-K report we provided earlier.

In [ ]:
import numpy as np

def retrieval(query):
    # 统一使用 embedding_model 变量名
    query_embedding = embedding_model.embed_query(query)

    query_embedding_np = np.array(query_embedding).astype("float32").reshape(1, -1)
    D, I = faiss_index.index.search(query_embedding_np, k=3)
    return [text_chunks[i] for i in I[0]]

###Test the retrieval function


In [ ]:
query = "Tell me something about Apple's new products and services?"
retrieved_docs = retrieval(query)
for doc in retrieved_docs:
    print(doc)

Air™, iPhone 17, iPhone 16 and iPhone 16e. Mac Mac ® is the Company’s line of personal computers based on its macOS ® operating system. The Mac line includes laptops MacBook Air ® and MacBook Pro ® , as well as desktops iMac ® , Mac mini ® , Mac Studio ® and Mac Pro ® . iPad iPad ® is the Company’s line of multipurpose tablets based on its iPadOS ® operating system. The iPad line includes iPad Pro ® , iPad Air ® , iPad and iPad mini ® . Wearables, Home and Accessories Wearables includes smartwatches, wireless headphones and spatial computers. The Company’s line of smartwatches, based on its watchOS ® operating system, includes Apple Watch ® Series 11, Apple Watch SE ® 3 and Apple Watch Ultra ® 3. The Company’s line of wireless headphones includes AirPods ® , AirPods Pro ® , AirPods Max ® and Beats ® products. Apple Vision Pro™ is the Company’s spatial computer based on its visionOS ® operating system. Home includes Apple TV 4K ® , the Company’s media streaming and gaming device based o

## Step 9: Augment the Prompt
- Create a function to augment user prompts with retrieved information for improved chatbot responses.

In [ ]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = retrieval(query)
    # get the text from the results
    source_knowledge = "\n".join(results)
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query. If the context does not contain enough information for you to answer the query, state it and you might provide general answer based on your knowledge.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

In [ ]:
print(augment_prompt(query))


Using the contexts below, answer the query. If the context does not contain enough information for you to answer the query, state it and you might provide general answer based on your knowledge.

    Contexts:
    Air™, iPhone 17, iPhone 16 and iPhone 16e. Mac Mac ® is the Company’s line of personal computers based on its macOS ® operating system. The Mac line includes laptops MacBook Air ® and MacBook Pro ® , as well as desktops iMac ® , Mac mini ® , Mac Studio ® and Mac Pro ® . iPad iPad ® is the Company’s line of multipurpose tablets based on its iPadOS ® operating system. The iPad line includes iPad Pro ® , iPad Air ® , iPad and iPad mini ® . Wearables, Home and Accessories Wearables includes smartwatches, wireless headphones and spatial computers. The Company’s line of smartwatches, based on its watchOS ® operating system, includes Apple Watch ® Series 11, Apple Watch SE ® 3 and Apple Watch Ultra ® 3. The Company’s line of wireless headphones includes AirPods ® , AirPods Pro ® , A

## Step 10: Test the Chatbot
Test your RAG chatbot with sample queries and observe its responses.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# 确保使用与 Step 6 一致的配置
chat = ChatOpenAI(
    model='gpt-4',
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    base_url="https://yinli.one/v1"
)

messages_RAG = [
    SystemMessage(content="You are a RAG chatbot designed to assist users with information from Apple's 2024 10-K report.")
]

query = "What are the key financial highlights of Apple Inc.?"
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages_RAG.append(prompt)

try:
    res_RAG = chat.invoke(messages_RAG)
    print(res_RAG.content)
except Exception as e:
    print(f"Chat Error: {e}")

Based on the provided contexts from Apple's 2024 10-K report, here are the key financial highlights for Apple Inc. for fiscal years 2023, 2024, and 2025:

**Net Sales:**
- 2025: $416,161 million
- 2024: $391,035 million
- 2023: $383,285 million

**Breakdown:**
- Products (2025): $307,003 million
- Services (2025): $109,158 million

**Gross Margin:**
- 2025: $195,201 million
- 2024: $180,683 million
- 2023: $169,148 million

**Operating Income:**
- 2025: $133,050 million
- 2024: $123,216 million
- 2023: $114,301 million

**Net Income:**
- 2025: $112,010 million
- 2024: $93,736 million
- 2023: $96,995 million

**Earnings per Share (EPS):**
- Basic (2025): $7.49
- Diluted (2025): $7.46

**Operating Expenses:**
- Research and Development (2025): $34,550 million
- Selling, General & Administrative (2025): $27,601 million
- Total Operating Expenses (2025): $62,151 million

**Cash Flow Highlights (2025):**
- Cash generated by operating activities: $111,482 million
- Cash generated by investin

## Step 11: Explore Langchain's RetrievalQA Chain
- Utilize Langchain's `RetrievalQA` chain for a simpler RAG implementation.

In [ ]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
import os

# Step 3: Set Up RetrievalQA (RAG Workflow)
def setup_rag(faiss_index):
    print("Setting up RetrievalQA...")
    retriever = faiss_index.as_retriever()
    # 确保此处配置与 Step 6/10 一致
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_key=os.environ.get("OPENAI_API_KEY"),
        base_url="https://yinli.one/v1"
    )
    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type="stuff")
    return qa_chain

In [ ]:
rag_pipeline = setup_rag(faiss_index)


Setting up RetrievalQA...


In [ ]:
rag_pipeline.invoke("How many full-time equivalent employees does Apple have?")

{'query': 'How many full-time equivalent employees does Apple have?',
 'result': "The information about the number of full-time equivalent employees at Apple is not provided in the given excerpts. For the most accurate and up-to-date details, you can refer to Apple's latest Annual Report on Form 10-K, which is available on their [Investor Relations website](https://investor.apple.com) or the SEC's website."}

# Summary and Observations
- Reflect on the performance of your RAG chatbot. How effectively did it answer your queries? Were there any limitations or areas where it struggled?
- Discuss the impact of using Retrieval Augmented Generation (RAG) in this scenario. What advantages does RAG offer compared to a traditional chatbot approach?
- Consider potential improvements or extensions for this project. Could you enhance the chatbot's capabilities by incorporating additional data sources or refining the retrieval process?

By completing this assignment, you have gained valuable experience in building a RAG-based chatbot using real-world data. This technology holds immense potential for various applications, such as customer support, information retrieval, and research assistance.

Congratulations on successfully completing this challenging and rewarding project!

